# 07 Alert System

In [2]:
import cv2
import time
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import subprocess


In [3]:
#model 
WEIGHTS = Path("runs/yolov8n_statefarm/weights/best.pt")#baseline best.pt

#video input set your video path here 
VIDEO_PATH = "/Users/cjarjun/Downloads/test_driver1.mp4" #change video path

#output
OUTPUT_DIR  = Path("../reports")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "alert_system_output.mp4"

#alert settings
IMG_SIZE          = 224
DISTRACTION_DELAY = 1.5 #seconds of distraction before alert triggers
SAFE_CLASS        = "c0"  #safe driving class

#Per-class confidence thresholds 
#c8(Hair/Makeup) fires too easily raised to 0.75 to reduce false positives
#c9(Talking to Passenger) is subtle  slightly raised too
CLASS_THRESHOLDS = {
    "c0": 0.35,  #Safe Driving
    "c1": 0.30,  #Texting (Right Hand)
    "c2": 0.30,  #Phone (Right Hand)
    "c3": 0.30,  #Texting (Left Hand)
    "c4": 0.30,  #Phone (Left Hand)
    "c5": 0.35,  #Operating Radio
    "c6": 0.35,  #Drinking
    "c7": 0.45,  #Reaching Behind
    "c8": 0.75,  #Hair/Makeup is prone to false positives, require higher confidence
    "c9": 0.45,  #Talking to Passenger
}

#temporal smoothing
SMOOTHING_FRAMES = 10   # look at last 10 frames
SMOOTHING_AGREE  = 0.6  # 60% must agree before prediction is accepted

#class labels
CLASS_INFO = {
    "c0": {"label": "Safe Driving",          "distracted": False},
    "c1": {"label": "Texting (Right Hand)",   "distracted": True},
    "c2": {"label": "Phone (Right Hand)",     "distracted": True},
    "c3": {"label": "Texting (Left Hand)",    "distracted": True},
    "c4": {"label": "Phone (Left Hand)",      "distracted": True},
    "c5": {"label": "Operating Radio",        "distracted": True},
    "c6": {"label": "Drinking",               "distracted": True},
    "c7": {"label": "Reaching Behind",        "distracted": True},
    "c8": {"label": "Hair / Makeup",          "distracted": True},
    "c9": {"label": "Talking to Passenger",   "distracted": True},
}


In [4]:
def beep():
    #macOS audio alert
    try:
        subprocess.run(["afplay", "/System/Library/Sounds/Funk.aiff"],
                       check=False, timeout=1,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
    except Exception:
        pass


def draw_frame(frame, class_key, conf, is_distracted,
               distraction_duration, alert_active):
 
    h, w = frame.shape[:2]
    info  = CLASS_INFO.get(class_key, CLASS_INFO["c0"])
    label = info["label"]

    #colour scheme
    if is_distracted:
        banner_color  = (43, 163, 250)      # red (BGR)
        text_color    = (255, 255, 255)
        status_text   = "⚠  DISTRACTED"
    else:
        banner_color  = (0, 180, 0)      # green
        text_color    = (255, 255, 255)
        status_text   = " SAFE"

    #status banner
    cv2.rectangle(frame, (0, 0), (w, 60), banner_color, -1)
    cv2.putText(frame, status_text, (15, 42),
                cv2.FONT_HERSHEY_DUPLEX, 1.2, text_color, 2)

    #class label & confidence
    cv2.rectangle(frame, (0, h - 160), (w, h), (30, 30, 30), -1)
    cv2.putText(frame, f"Behaviour : {label}",
                (15, h - 65),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (200, 200, 200), 2)
    cv2.putText(frame, f"Confidence: {conf*100:.1f}%",
                (15, h - 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)

    #distraction duration counter
    if is_distracted and distraction_duration > 0:
        dur_text = f"Distracted for: {distraction_duration:.1f}s"
        cv2.putText(frame, dur_text, (w - 340, h - 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 100, 255), 2)

    #flashing alert 
    if alert_active:
        #semi transparent red overlay on full frame
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 180), -1)
        cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)

        #ALERT text in centre
        alert_text = "DRIVER ALERT!"
        text_size  = cv2.getTextSize(alert_text,
                                     cv2.FONT_HERSHEY_DUPLEX, 1.5, 3)[0]
        text_x = (w - text_size[0]) // 2
        cv2.putText(frame, alert_text, (text_x, h // 2),
                    cv2.FONT_HERSHEY_DUPLEX, 1.5, (43, 163, 250), 3)

    return frame


In [ ]:
from collections import deque, Counter

def run_alert_system(source, output_path=None, show_window=True):
 
    model = YOLO(str(WEIGHTS))

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f" could not open source: {source}")
        return

    #video properties
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    source_name = "Webcam" if source == 0 else Path(str(source)).name

    if total > 0:
    #video writer
     writer = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    #state tracking 
    prediction_buffer = deque(maxlen=SMOOTHING_FRAMES)  # majority vote buffer
    distraction_start = None
    last_beep_time    = 0
    beep_cooldown     = 3.0
    frame_count       = 0
    distracted_frames = 0
    total_frames      = 0

    print("\nstarting alert system press 'Q' to quit.\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count  += 1
        total_frames += 1
        current_time  = time.time()

        #run inference
        result    = model.predict(frame, imgsz=IMG_SIZE, verbose=False)[0]
        class_idx = result.probs.top1
        conf      = float(result.probs.top1conf)
        raw_class = result.names[class_idx] #raw prediction this frame
        #if confidence doesn't meet this class's threshold then it treat as safe
        threshold = CLASS_THRESHOLDS.get(raw_class, 0.45)
        gated_class = raw_class if conf >= threshold else "c0"

        #temporal majority vote
        prediction_buffer.append(gated_class)
        votes        = Counter(prediction_buffer)
        top_class, top_votes = votes.most_common(1)[0]
        agreement    = top_votes / len(prediction_buffer)

        #only predict if enough frames agree
        if agreement >= SMOOTHING_AGREE:
            class_key = top_class
        else:
            class_key = "c0"  #not sure show as safe

        #determine if distracted
        is_distracted = CLASS_INFO[class_key]["distracted"]

        #track distraction duration
        if is_distracted:
            distracted_frames += 1
            if distraction_start is None:
                distraction_start = current_time
            distraction_duration = current_time - distraction_start
        else:
            distraction_start    = None
            distraction_duration = 0.0

        #trigger alert after delay
        alert_active = (
            is_distracted and
            distraction_duration >= DISTRACTION_DELAY
        )

        #auudio beep with cooldown
        if alert_active and (current_time - last_beep_time) > beep_cooldown:
            beep()
            last_beep_time = current_time

        #draw overlays
        frame = draw_frame(
            frame, class_key, conf,
            is_distracted, distraction_duration, alert_active
        )

        #write frame
        if writer:
            writer.write(frame)

        #show live window 
        if show_window:
            cv2.imshow("Distracted Driver Alert System", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                print("Stopped by user.")
                break

        #progress print every 50 frames
        if frame_count % 50 == 0:
            pct = f"{frame_count/total*100:.1f}%" if total > 0 else f"{frame_count} frames"
            print(f"  Processed {frame_count} frames ({pct}) | "
                  f"Smoothed: {CLASS_INFO[class_key]['label']} | "
                  f"Raw: {CLASS_INFO[raw_class]['label']} ({conf*100:.1f}%)")

    #cleanup
    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()

    #final stats
    distraction_pct = (distracted_frames / total_frames * 100) if total_frames > 0 else 0



In [6]:

run_alert_system(
    source=VIDEO_PATH,
    output_path=OUTPUT_VIDEO,
    show_window=True        # set False to run headless (no popup window)
)


starting alert system press 'Q' to quit.

  Processed 50 frames (11.2%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (99.1%)
  Processed 100 frames (22.5%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (90.4%)
  Processed 150 frames (33.7%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (95.2%)
  Processed 200 frames (44.9%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (99.5%)
  Processed 250 frames (56.2%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (99.6%)
  Processed 300 frames (67.4%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (99.9%)
  Processed 350 frames (78.7%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (100.0%)
  Processed 400 frames (89.9%) | Smoothed: Phone (Right Hand) | Raw: Phone (Right Hand) (99.9%)


In [ ]:

WEBCAM_OUTPUT = OUTPUT_DIR / "webcam_output.mp4"

run_alert_system(
    source=0,                     # 0 = default webcam
    output_path=WEBCAM_OUTPUT,
    show_window=True
)


starting alert system press 'Q' to quit.

  Processed 50 frames (50 frames) | Smoothed: Hair / Makeup | Raw: Hair / Makeup (63.5%)
  Processed 100 frames (100 frames) | Smoothed: Hair / Makeup | Raw: Hair / Makeup (100.0%)
  Processed 150 frames (150 frames) | Smoothed: Hair / Makeup | Raw: Hair / Makeup (96.3%)
  Processed 200 frames (200 frames) | Smoothed: Safe Driving | Raw: Drinking (44.3%)
  Processed 250 frames (250 frames) | Smoothed: Hair / Makeup | Raw: Hair / Makeup (84.5%)
  Processed 300 frames (300 frames) | Smoothed: Safe Driving | Raw: Hair / Makeup (93.5%)
  Processed 350 frames (350 frames) | Smoothed: Talking to Passenger | Raw: Talking to Passenger (71.5%)
  Processed 400 frames (400 frames) | Smoothed: Hair / Makeup | Raw: Hair / Makeup (97.5%)
Stopped by user.


: 